# Baseline 2.0 — Three-class YOLO26n

This experiment keeps only the categories named `3`, `5`, and `6` from `data.yaml`. Original YOLO class IDs `2`, `3`, and `4` are remapped to new IDs `0`, `1`, and `2`. The source dataset is left unchanged; a filtered copy is created under `datasets/baseline2.0`.

Run the cells from top to bottom. Training uses 100 epochs and resumes from this experiment's `last.pt` checkpoint when available.

In [ ]:
%pip install ultralytics pyyaml

In [1]:
from collections import Counter
from pathlib import Path
import shutil
import yaml

ROOT = Path.cwd()
if not (ROOT / "data.yaml").is_file() and (ROOT.parent / "data.yaml").is_file():
    ROOT = ROOT.parent

SOURCE_YAML = ROOT / "data.yaml"
if not SOURCE_YAML.is_file():
    raise FileNotFoundError(f"Could not find data.yaml from {Path.cwd()}")

with SOURCE_YAML.open(encoding="utf-8") as f:
    source_config = yaml.safe_load(f)

CLASS_REMAP = {2: 0, 3: 1, 4: 2}
CLASS_NAMES = ["3", "5", "6"]
OUTPUT_DATA = ROOT / "datasets" / "baseline2.0"
split_dirs = {
    "train": ("train", "train"),
    "val": ("valid", "val"),
    "test": ("test", "test"),
}
class_counts = {}
for yaml_split, (output_split, _) in split_dirs.items():
    source_images = Path(source_config[yaml_split])
    if not source_images.is_absolute():
        source_images = SOURCE_YAML.parent / source_images
    source_labels = source_images.parent / "labels"
    output_images = OUTPUT_DATA / output_split / "images"
    output_labels = OUTPUT_DATA / output_split / "labels"
    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)
    class_counts[yaml_split] = Counter()

    images = sorted(p for p in source_images.iterdir() if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    if not images:
        raise FileNotFoundError(f"No images found in {source_images}")

    for image_path in images:
        source_label = source_labels / f"{image_path.stem}.txt"
        if not source_label.is_file():
            raise FileNotFoundError(f"Missing label file for {image_path.name}: {source_label}")

        shutil.copy2(image_path, output_images / image_path.name)
        filtered_rows = []
        for line in source_label.read_text(encoding="utf-8").splitlines():
            fields = line.split()
            if not fields:
                continue
            old_id = int(fields[0])
            if old_id not in range(len(source_config["names"])):
                raise ValueError(f"Invalid class ID {old_id} in {source_label}")
            if old_id in CLASS_REMAP:
                new_id = CLASS_REMAP[old_id]
                fields[0] = str(new_id)
                filtered_rows.append(" ".join(fields))
                class_counts[yaml_split][new_id] += 1

        destination_label = output_labels / source_label.name
        destination_label.write_text("\n".join(filtered_rows) + ("\n" if filtered_rows else ""), encoding="utf-8")

dataset_yaml = {
    "path": str(OUTPUT_DATA),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}
SUBSET_YAML = OUTPUT_DATA / "data.yaml"
SUBSET_YAML.parent.mkdir(parents=True, exist_ok=True)
SUBSET_YAML.write_text(yaml.safe_dump(dataset_yaml, sort_keys=False), encoding="utf-8")

for split, counts in class_counts.items():
    print(f"{split} instances: " + ", ".join(f"{CLASS_NAMES[class_id]}={counts[class_id]}" for class_id in range(3)))
print(f"Filtered dataset YAML: {SUBSET_YAML}")

train instances: 3=3027, 5=32637, 6=3576
val instances: 3=761, 5=7428, 6=1336
test instances: 3=438, 5=4135, 6=639
Filtered dataset YAML: d:\AI-TOD.v2i.yolo26\datasets\baseline2.0\data.yaml


## Train and resume

This run is independent of the original five-class baseline. If interrupted, rerun this cell to resume from the latest completed epoch in `runs/detect/baseline2.0/weights/last.pt`.

In [2]:
import torch
from ultralytics import YOLO

EPOCHS = 100
DEVICE = 0 if torch.cuda.is_available() else "cpu"
RUN_DIR = ROOT / "runs" / "detect" / "baseline2.0"
LAST_CHECKPOINT = RUN_DIR / "weights" / "last.pt"

if LAST_CHECKPOINT.is_file():
    print(f"Resuming from {LAST_CHECKPOINT}")
    model = YOLO(str(LAST_CHECKPOINT))
    train_results = model.train(
        resume=True,
        epochs=EPOCHS,
        data=str(SUBSET_YAML),
        device=DEVICE,
    )
else:
    print("No checkpoint found; starting from pretrained yolo26n.pt")
    model = YOLO("yolo26n.pt")
    train_results = model.train(
        data=str(SUBSET_YAML),
        epochs=EPOCHS,
        imgsz=640,
        batch=8,
        device=DEVICE,
        workers=0,
        project=str(ROOT / "runs" / "detect"),
        name="baseline2.0",
        exist_ok=True,
        patience=10,
        seed=42,
        plots=True,
    )
print(f"Training outputs: {train_results.save_dir}")

Resuming from d:\AI-TOD.v2i.yolo26\runs\detect\baseline2.0\weights\last.pt
New https://pypi.org/project/ultralytics/8.4.215 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.211  Python-3.13.3 torch-2.15.0+cpu CPU (13th Gen Intel Core i5-1335U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\AI-TOD.v2i.yolo26\datasets\baseline2.0\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=

## Evaluate on the filtered test split

Run this after training completes. The test labels use the same three-class filtering and ID remapping as train and validation.

In [3]:
best_weights = Path(train_results.save_dir) / "weights" / "best.pt"
if not best_weights.is_file():
    raise FileNotFoundError(f"Best checkpoint was not created: {best_weights}")

best_model = YOLO(str(best_weights))
test_metrics = best_model.val(
    data=str(SUBSET_YAML),
    split="test",
    imgsz=640,
    batch=8,
    device=DEVICE,
    workers=0,
)
print(test_metrics.results_dict)

Ultralytics 8.4.211  Python-3.13.3 torch-2.15.0+cpu CPU (13th Gen Intel Core i5-1335U)
YOLO26n summary (fused): 121 layers, 2,390,221 parameters, 0 gradients, 5.4 GFLOPs
WARNING val: Slow image access detected (ping: 0.2±0.0 ms, read: 3.9±1.8 MB/s, size: 37.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning D:\AI-TOD.v2i.yolo26\datasets\baseline2.0\test\labels... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 303.0it/s 1.0s0.1s
val: New cache created: D:\AI-TOD.v2i.yolo26\datasets\baseline2.0\test\labels.cache
WARNING Dataset images contain up to 355 objects (test=355), but max_det=300. This mismatch can cap recall and produce invalid validation metrics. Raising it may increase validation cost but cannot increase model or export capacity, which may cap recall. Setting max_det=355 to match the observed maximum.
                 Class     Images  Instances      Box(P